# Phase 12: Unseen / Zero-Day Cyberattack Detection
### Smart Grid Cyber-Physical Intrusion Detection

**The Research Goal:**
Determine whether an Intrusion Detection System can recognize a completely unseen attack class as anomalous rather than incorrectly classifying it as normal or assigning it to a known class.

**Experimental Setup:**
- **Training Set**: Benign + Known Attacks (`industroyer_modbus`, `mqtt_price_signal_manipulation`).
- **Held-Out Zero-Day Attack**: `mitm_false_data_injection` (MitM FDI).
- **Rule Enforcement**: The held-out attack is NEVER used during training, scaling, feature selection, or threshold tuning!


In [ ]:
import os
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import torch.nn as nn
from sklearn.metrics import accuracy_score, f1_score, recall_score

if os.path.exists('src'):
    PROJECT_ROOT = os.path.abspath('.')
elif os.path.exists('../src'):
    PROJECT_ROOT = os.path.abspath('..')
else:
    PROJECT_ROOT = os.path.abspath('.')
sys.path.insert(0, PROJECT_ROOT)

from src.data_loader import load_data4cyber, get_preprocessor

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
sns.set_theme(style="whitegrid")


## 1. Zero-Day Partitioning


In [ ]:
data = load_data4cyber(random_state=42)
prep = get_preprocessor(scaling=True)

X_tr_sc = prep.fit_transform(data['X_train'])
X_te_sc = prep.transform(data['X_test'])

le = data['label_encoder']
unseen_name = 'mitm_false_data_injection'
unseen_idx = list(le.classes_).index(unseen_name)
benign_idx = list(le.classes_).index('benign')

print(f"Held-out Unseen Zero-Day Attack: {unseen_name}")

# Normal training data
X_norm_tr = X_tr_sc[data['y_train_family'] == benign_idx]

# Split test sets
X_te_norm = X_te_sc[data['y_test_family'] == benign_idx]
X_te_known = X_te_sc[(data['y_test_family'] != benign_idx) & (data['y_test_family'] != unseen_idx)]
X_te_unseen = X_te_sc[data['y_test_family'] == unseen_idx]

print(f"Test Normal Samples: {len(X_te_norm)}")
print(f"Test Known Attack Samples: {len(X_te_known)}")
print(f"Test Held-Out Zero-Day Attack Samples: {len(X_te_unseen)}")


## 2. Train Autoencoder on Normal Baseline


In [ ]:
class GridAutoencoder(nn.Module):
    def __init__(self, in_features, latent_dim=16):
        super(GridAutoencoder, self).__init__()
        self.encoder = nn.Sequential(
            nn.Linear(in_features, 64),
            nn.BatchNorm1d(64),
            nn.LeakyReLU(0.1),
            nn.Linear(64, latent_dim),
            nn.LeakyReLU(0.1)
        )
        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, 64),
            nn.BatchNorm1d(64),
            nn.LeakyReLU(0.1),
            nn.Linear(64, in_features)
        )
    def forward(self, x):
        return self.decoder(self.encoder(x))

ae = GridAutoencoder(X_norm_tr.shape[1], latent_dim=16).to(DEVICE)
crit = nn.MSELoss()
opt = torch.optim.Adam(ae.parameters(), lr=1e-3)

tx_tr = torch.tensor(X_norm_tr, dtype=torch.float32)
loader = torch.utils.data.DataLoader(torch.utils.data.TensorDataset(tx_tr), batch_size=64, shuffle=True)

ae.train()
for ep in range(20):
    for (bx,) in loader:
        bx = bx.to(DEVICE)
        opt.zero_grad()
        loss = crit(ae(bx), bx)
        loss.backward()
        opt.step()

ae.eval()
with torch.no_grad():
    recon_tr = ae(tx_tr.to(DEVICE))
    err_tr = torch.mean((tx_tr.to(DEVICE) - recon_tr)**2, dim=1).cpu().numpy()

threshold = np.percentile(err_tr, 95.0)
print(f"Anomaly Detection Threshold: {threshold:.5f}")


## 3. Zero-Day Evaluation: Can the Autoencoder Detect the Unseen Attack?


In [ ]:
with torch.no_grad():
    def get_err(X):
        tx = torch.tensor(X, dtype=torch.float32).to(DEVICE)
        return torch.mean((tx - ae(tx))**2, dim=1).cpu().numpy()

err_norm = get_err(X_te_norm)
err_known = get_err(X_te_known)
err_unseen = get_err(X_te_unseen)

unseen_det_rate = np.mean(err_unseen >= threshold)
known_det_rate = np.mean(err_known >= threshold)
false_alarm_rate = np.mean(err_norm >= threshold)

print("="*60)
print(f"ZERO-DAY ATTACK EVALUATION ({unseen_name}):")
print(f"  Unseen Attack Detection Rate: {unseen_det_rate * 100:.2f}%")
print(f"  Known Attack Detection Rate:  {known_det_rate * 100:.2f}%")
print(f"  False Alarm Rate on Normal:   {false_alarm_rate * 100:.2f}%")
print("="*60)


## 4. Visualizing Zero-Day Separation vs. Normal Operations


In [ ]:
plt.figure(figsize=(10, 5))
sns.kdeplot(err_norm, label='Normal Telemetry', color='green', fill=True, common_norm=False)
sns.kdeplot(err_known, label='Known Attacks (Modbus/MQTT)', color='orange', fill=True, common_norm=False)
sns.kdeplot(err_unseen, label=f'Unseen Zero-Day ({unseen_name})', color='red', fill=True, common_norm=False)
plt.axvline(threshold, color='black', linestyle='--', linewidth=2, label=f'Threshold ({threshold:.4f})')
plt.title(f"Zero-Day Detection: Reconstruction Deviation on Unseen {unseen_name}", fontweight='bold')
plt.xlabel("Reconstruction Residual Error")
plt.ylabel("Density")
plt.xlim(0, np.percentile(err_unseen, 95))
plt.legend()
plt.tight_layout()
plt.show()
